# Jour 2 : Les data pour le Machine Learning, collecte et preprocessing


## Phase 0 : Récupérer la donnée et l'ouvrir

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Forme :", df.shape)
print(df.dtypes)
df.head()

Forme : (7043, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Phase 1 : L'audit qualité

In [2]:
def audit_qualite(df):
    """Affiche un rapport de santé du dataset.
    Doit montrer : forme, types, % de manquants par colonne (triés),
    et la répartition de la cible Churn (en valeur ET en pourcentage).
    """
    # afficher la forme (lignes, colonnes)
    print(f"Forme : {df.shape}")
    print()

    # pourcentage de valeurs manquantes par colonne, trié décroissant
    manquants = df.isna().sum()
    pourcentage = (df.isna().mean() * 100).round(1)
    resume = pd.DataFrame({"manquants": manquants, "pourcent": pourcentage})
    resume_filtre = resume[resume["manquants"] > 0].sort_values("pourcent", ascending=False)

    if len(resume_filtre) == 0:
        print("Manquants détectés : 0 colonne  (méfiance : des trous sont peut-être cachés, voir Phase 2)")
    else:
        print("Colonnes avec manquants :")
        print(resume_filtre)
    print()

    # répartition de la cible Churn en nombre et en %
    churn_counts = df["Churn"].value_counts()
    churn_pct = (df["Churn"].value_counts(normalize=True) * 100).round(1)
    print("Répartition de la cible Churn :")
    for val in churn_counts.index:
        print(f"  Churn {val} : {churn_counts[val]} ({churn_pct[val]}%)")

In [4]:
# cas normal
print("=== CAS NORMAL ===")
audit_qualite(df)

# cas limite : une seule classe
print("\n=== CAS LIMITE (Churn == 'No' seulement) ===")
audit_qualite(df[df["Churn"] == "No"])

# cas adversarial : déséquilibre visible ?
print("\n=== CAS ADVERSARIAL ===")
print("Le déséquilibre 73/27 saute-t-il aux yeux ? :")
churn_pct = (df["Churn"].value_counts(normalize=True) * 100).round(1)
print(churn_pct)
seuil = 60
if churn_pct.max() > seuil:
    print(f" Déséquilibre détecté ! La classe majoritaire dépasse {seuil}%")

=== CAS NORMAL ===
Forme : (7043, 21)

Manquants détectés : 0 colonne  (méfiance : des trous sont peut-être cachés, voir Phase 2)

Répartition de la cible Churn :
  Churn No : 5174 (73.5%)
  Churn Yes : 1869 (26.5%)

=== CAS LIMITE (Churn == 'No' seulement) ===
Forme : (5174, 21)

Manquants détectés : 0 colonne  (méfiance : des trous sont peut-être cachés, voir Phase 2)

Répartition de la cible Churn :
  Churn No : 5174 (100.0%)

=== CAS ADVERSARIAL ===
Le déséquilibre 73/27 saute-t-il aux yeux ? :
Churn
No     73.5
Yes    26.5
Name: proportion, dtype: float64
 Déséquilibre détecté ! La classe majoritaire dépasse 60%


## Phase 2 : La colonne piégée (types incohérents et trous cachés)


In [7]:
def reparer_total_charges(df):
    """Convertit TotalCharges en numérique et traite les trous révélés.
    Doit renvoyer le df réparé et afficher combien de trous ont été démasqués.
    """
    df = df.copy()

    # vérifier que la colonne n'est pas 100% non numérique
    test = pd.to_numeric(df["TotalCharges"], errors="coerce")
    taux_nan = test.isna().mean()
    if taux_nan > 0.95:
        print(" Colonne quasi-entièrement non numérique, conversion refusée.")
        return df

    # convertir en numérique (les espaces " " deviennent NaN)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # compter les trous révélés
    nb_trous = df["TotalCharges"].isna().sum()
    print(f"Trous démasqués après conversion : {nb_trous}")
    print(f"Type après conversion : {df['TotalCharges'].dtype}")

    # imputer par la médiane
    mediane = df["TotalCharges"].median()
    df["TotalCharges"] = df["TotalCharges"].fillna(mediane)
    print(f"Trous après imputation : {df['TotalCharges'].isna().sum()}")
    print(f"Valeur utilisée (médiane) : {mediane:.2f}")

    return df

In [8]:
# happy path
print("=== HAPPY PATH ===")
df = reparer_total_charges(df)

# edge case : colonne 100% texte
print("\n=== EDGE CASE (colonne 100% texte) ===")
df_test = df.copy()
df_test["TotalCharges"] = "abc"
reparer_total_charges(df_test)

# adversarial : virgule au lieu du point
print("\n=== ADVERSARIAL (virgule au lieu du point) ===")
df_test2 = df.copy()
df_test2["TotalCharges"] = df_test2["TotalCharges"].astype(str).str.replace(".", ",", regex=False)
test_corrompu = pd.to_numeric(df_test2["TotalCharges"], errors="coerce")
print(f"NaN créés par la virgule : {test_corrompu.isna().sum()}")
print("→ Solution : détecter avec .str.contains(',') avant conversion")

=== HAPPY PATH ===
Trous démasqués après conversion : 0
Type après conversion : float64
Trous après imputation : 0
Valeur utilisée (médiane) : 1397.47

=== EDGE CASE (colonne 100% texte) ===
 Colonne quasi-entièrement non numérique, conversion refusée.

=== ADVERSARIAL (virgule au lieu du point) ===
NaN créés par la virgule : 7043
→ Solution : détecter avec .str.contains(',') avant conversion


## Phase 3 : Encoder les catégorielles

In [ ]:
def encoder_features(df):
    """Encode toutes les colonnes catégorielles.
    Doit renvoyer un df 100% numérique, prêt pour un modèle.
    """
    df = df.copy()

    # supprimer customerID 
    df = df.drop(columns=["customerID"])

    # encoder la cible Churn : Yes=1, No=0
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    # colonnes binaires Yes/No : encodage simple 0/1
    colonnes_binaires = [
        "gender", "Partner", "Dependents", "PhoneService",
        "PaperlessBilling", "MultipleLines", "OnlineSecurity",
        "OnlineBackup", "DeviceProtection", "TechSupport",
        "StreamingTV", "StreamingMovies"
    ]
    mapping_binaire = {"Yes": 1, "No": 0, "Male": 1, "Female": 0,
                       "No phone service": 0, "No internet service": 0}
    for col in colonnes_binaires:
        if col in df.columns:
            df[col] = df[col].map(mapping_binaire)

    # Contract : ordinal (Month-to-month < One year < Two year)
    from sklearn.preprocessing import OrdinalEncoder
    ordre_contract = [["Month-to-month", "One year", "Two year"]]
    enc = OrdinalEncoder(categories=ordre_contract)
    df["Contract"] = enc.fit_transform(df[["Contract"]])

    # colonnes nominales restantes : One-Hot
    colonnes_nominales = ["InternetService", "PaymentMethod"]
    df = pd.get_dummies(df, columns=colonnes_nominales, drop_first=False)

    print(f"Colonnes avant : 21")
    print(f"Colonnes après : {df.shape[1]}")
    print(f"Types restants non numériques : {list(df.select_dtypes('object').columns)}")
    print(f"\nAperçu des nouvelles colonnes :\n{list(df.columns)}")

    return df

In [10]:
# happy path
print("=== HAPPY PATH ===")
df = encoder_features(df)
print(f"\nDataset 100% numérique : {df.select_dtypes('object').shape[1] == 0}")

# edge case : Contract est-il ordinal ou nominal ?
print("\n=== EDGE CASE : Contract ===")
print("Contract encodé en ordinal (0=month-to-month, 1=one year, 2=two year)")
print(df["Contract"].value_counts().sort_index())

# adversarial : customerID encodé en One-Hot = explosion
print("\n=== ADVERSARIAL : customerID en One-Hot ===")
df_test = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
nb_clients_uniques = df_test["customerID"].nunique()
print(f"Si customerID encodé en One-Hot : {nb_clients_uniques} colonnes créées !")
print("→ C'est l'explosion de dimensions. On le supprime toujours.")

=== HAPPY PATH ===
Colonnes avant : 21
Colonnes après : 25
Types restants non numériques : []

Aperçu des nouvelles colonnes :
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

Dataset 100% numérique : True

=== EDGE CASE : Contract ===
Contract encodé en ordinal (0=month-to-month, 1=one year, 2=two year)
Contract
0.0    3875
1.0    1473
2.0    1695
Name: count, dtype: int64

=== ADVERSARIAL : customerID en One-Hot ===
Si customerID encodé en One-Hot : 7043 colonnes créées !
→ C'est l'explosion de dimensions. On le supprime toujours.


## Phase 4 : Traiter les valeurs aberrantes

## Phase 5 : Corrélations et chasse à la multicolinéarité

## Phase 6 : Les variables qui prédisent vraiment le churn


## Phase 7 : Split, scaling, et le grand piège de la fuite

## Phase 8 : Le bilan, et jusqu'où on peut pousser